# Taller: Importación y Extracción de Datos

**Inteligencia Artificial con Aplicaciones en Economía I**
🎓 Universidad Externado de Colombia — Facultad de Economía

Integrantes: *JUAN MANUEL VELLAIZAN, MARIA PAULA CARO ORZOCO*


## Instalaciones necesarias

In [6]:
# Para poder leer y abrir archivos de Excel
!pip install xlrd openpyxl

In [7]:
# Para trabajar datos tabulares
!pip install pandas

In [8]:
# Para conectarnos a la API del Banco Mundial
!pip install wbdata

## Importar librerías

In [9]:
import numpy as np
import pandas as pd
import requests
import os
import datetime
import wbdata

### Mejorar visualización de los dataframes

In [10]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

### Función auxiliar de exploración

Para no repetir 8 veces el mismo bloque de código, se define una función que aplica exactamente lo pedido en el enunciado: primeras 5 filas, últimas 10 filas, 3 filas aleatorias, filas 2 a 5 con solo la segunda columna, `len()`, `.shape`, `.columns`, `.dtypes` e `.info()`.

In [11]:
def explorar_dataset(df, nombre="Dataset"):
    print("=" * 80)
    print(f"EXPLORACIÓN: {nombre}")
    print("=" * 80)

    print("\nPrimeras 5 filas:")
    display(df.head())

    print("\nÚltimas 10 filas:")
    display(df.tail(10))

    print("\n3 filas aleatorias:")
    display(df.sample(min(3, len(df))))

    print("\nFilas 2 a 5, solo la segunda columna:")
    display(df.iloc[1:5, [1]])

    print("\nlen(df):", len(df))

    print("\ndf.shape:", df.shape)

    print("\ndf.columns:")
    print(df.columns.tolist())

    print("\ndf.dtypes:")
    print(df.dtypes)

    print("\ndf.info():")
    df.info()
    print()

## 1. Base de países del WEO (FMI) — importada desde Google Drive

**Pasos manuales (antes de correr esta celda):**
1. Descargar la base "WEO by countries" en https://www.imf.org/en/Publications/WEO/weo-database (queda como texto delimitado por tabulaciones, aunque la extensión sea `.xls`).
2. Subir el archivo a una carpeta de Google Drive, p. ej. `MiDrive/Econometria/`.
3. Conectar el notebook a Google Drive con `drive.mount`, tal como se hizo en clase.

In [12]:
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Para establecer el directorio donde quedó el archivo del WEO
os.chdir('/content/drive/MyDrive/taller ia 1')
os.listdir('.')

['base datos taller 1 .xlsx']

In [16]:
df_weo = pd.read_excel('base datos taller 1 .xlsx', na_values=["n/a", "--"])
df_weo.head()

,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
0,512,AFG,NGDP_R,Afghanistan,"Gross domestic product, constant prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,453484,492903,496209,554910,584658,662650,688247,829924,899956,958266,"1,092.118","1,154.178","1,185.306","1,197.012","1,222.917","1,255.288","1,270.216","1,319.902","1,288.869","1,101.445","1,032.712","1,056.123",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
1,512,AFG,NGDP_RPCH,Afghanistan,"Gross domestic product, constant prices",Annual percentages of constant price GDP are y...,Percent change,Units,"See notes for: Gross domestic product, consta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8692,0.671,11830,5361,13340,3863,20585,8438,6479,13968,5683,2697,0.988,2164,2647,1189,3912,-2351,-14542,-6240,2267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
2,512,AFG,NGDP,Afghanistan,"Gross domestic product, current prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178756,220013,246210,304926,345817,427495,517509,607227,711759,836222,"1,033.591","1,116.827","1,183.039","1,226.570","1,222.917","1,285.460","1,327.690","1,469.596","1,547.289","1,251.172","1,283.442","1,350.910",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
3,512,AFG,NGDPD,Afghanistan,"Gross domestic product, current prices",Values are based upon GDP in national currency...,U.S. dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4367,4553,5146,6167,6925,8556,10297,12066,15325,17890,20293,20170,20616,20057,18020,18883,18336,18876,20136,14278,14501,17248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
4,512,AFG,PPPGDP,Afghanistan,"Gross domestic product, current prices",These data form the basis for the country weig...,Purchasing power parity; international dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22742,25206,26058,30054,32642,37999,40227,48807,53568,58216,67583,72639,75898,77358,79784,83362,89369,97801,100898,85768,86149,91272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"


In [17]:
explorar_dataset(df_weo, "Base de países WEO (FMI)")

EXPLORACIÓN: Base de países WEO (FMI)

Primeras 5 filas:


,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
0,512,AFG,NGDP_R,Afghanistan,"Gross domestic product, constant prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,453484,492903,496209,554910,584658,662650,688247,829924,899956,958266,"1,092.118","1,154.178","1,185.306","1,197.012","1,222.917","1,255.288","1,270.216","1,319.902","1,288.869","1,101.445","1,032.712","1,056.123",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
1,512,AFG,NGDP_RPCH,Afghanistan,"Gross domestic product, constant prices",Annual percentages of constant price GDP are y...,Percent change,Units,"See notes for: Gross domestic product, consta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8692,0.671,11830,5361,13340,3863,20585,8438,6479,13968,5683,2697,0.988,2164,2647,1189,3912,-2351,-14542,-6240,2267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
2,512,AFG,NGDP,Afghanistan,"Gross domestic product, current prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178756,220013,246210,304926,345817,427495,517509,607227,711759,836222,"1,033.591","1,116.827","1,183.039","1,226.570","1,222.917","1,285.460","1,327.690","1,469.596","1,547.289","1,251.172","1,283.442","1,350.910",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
3,512,AFG,NGDPD,Afghanistan,"Gross domestic product, current prices",Values are based upon GDP in national currency...,U.S. dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4367,4553,5146,6167,6925,8556,10297,12066,15325,17890,20293,20170,20616,20057,18020,18883,18336,18876,20136,14278,14501,17248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
4,512,AFG,PPPGDP,Afghanistan,"Gross domestic product, current prices",These data form the basis for the country weig...,Purchasing power parity; international dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22742,25206,26058,30054,32642,37999,40227,48807,53568,58216,67583,72639,75898,77358,79784,83362,89369,97801,100898,85768,86149,91272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"



Últimas 10 filas:


,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
8616,698,ZWE,GGXONLB_NGDP,Zimbabwe,General government primary net lending/borrowing,Primary net lending/borrowing is net lending (...,Percent of GDP,Units,See notes for: General government primary net...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2599,0.147,-0.911,0.290,-0.010,0.790,-2174,0.056,-0.360,-0.324,-1149,-6004,-9434,-4573,-2013,0.090,-2573,-4695,-5198,-1096,0.537,1529,1948,2028,2108,2108,"2,024.00"
8617,698,ZWE,GGXWDN,Zimbabwe,General government net debt,Net debt is calculated as gross debt minus fin...,National currency,Billions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8618,698,ZWE,GGXWDN_NGDP,Zimbabwe,General government net debt,Net debt is calculated as gross debt minus fin...,Percent of GDP,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8619,698,ZWE,GGXWDG,Zimbabwe,General government gross debt,Gross debt consists of all liabilities that re...,National currency,Billions,Source: Ministry of Finance or Treasury Latest...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001,0.001,0.001,0.002,0.002,0.002,0.002,0.003,0.003,0.003,0.004,0.004,0.006,0.007,0.070,0.466,0.743,4935,51682,557793,673344,739670,786902,862793,930773,993329,"2,024.00"
8620,698,ZWE,GGXWDG_NGDP,Zimbabwe,General government gross debt,Gross debt consists of all liabilities that re...,Percent of GDP,Units,See notes for: General government gross debt ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33062,39438,44671,61078,58710,47578,42858,38379,37048,42312,48029,49908,68935,48125,82337,84466,58172,99536,96586,94587,58568,56051,53294,52315,50485,48312,"2,024.00"
8621,698,ZWE,NGDP_FY,Zimbabwe,Gross domestic product corresponding to fiscal...,Gross domestic product corresponding to fiscal...,National currency,Billions,Source: Ministry of Finance or Treasury Latest...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004,0.004,0.003,0.003,0.003,0.003,0.004,0.004,0.005,0.005,0.005,0.004,0.004,0.004,0.004,0.004,0.003,0.003,0.003,0.004,0.005,0.006,0.007,0.008,0.008,0.008,0.008,0.009,0.015,0.085,0.552,1277,4958,53509,589716,"1,149.685","1,319.643","1,476.529","1,649.212","1,843.656","2,056.061","2,024.00"
8622,698,ZWE,BCA,Zimbabwe,Current account balance,Current account is all transactions other than...,U.S. dollars,Billions,Source: Reserve Bank of Zimbabwe and Ministry ...,-0.301,-0.674,-0.748,-0.504,-0.171,-0.153,-0.051,0.000,0.050,-0.079,-0.257,-0.547,-0.842,-0.311,-0.318,-0.369,-0.180,-0.801,-0.159,0.244,0.322,0.365,0.249,-0.069,-0.128,-0.087,0.154,0.298,-0.138,-0.730,-1655,-2750,-2278,-2649,-2334,-1597,-0.697,-0.271,-1380,0.920,0.678,0.348,0.305,0.135,0.505,1143,1210,1198,1226,1120,0.996,"2,023.00"
8623,698,ZWE,BCA_NGDPD,Zimbabwe,Current account balance,Current account is all transactions other than...,Percent of GDP,Units,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2534,-5789,-10804,-4109,-3995,-4466,-1779,-7717,-1312,2070,2838,3248,2323,-0.720,-1353,-0.958,1887,3829,-2061,-7557,-13747,-19497,-13308,-13874,-11971,-7998,-3394,-1230,-3737,3536,2524,0.967,0.935,0.384,1435,2994,3063,2927,2895,2554,2196,"2,023.00"
8624,NaN,NaN,Na


3 filas aleatorias:


,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
147,171,AND,NGSD_NGDP,Andorra,Gross national savings,Expressed as a ratio of gross national savings...,Percent of GDP,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7027,733,SSD,GGXCNL,South Sudan,General government net lending/borrowing,Net lending (+)/ borrowing (-) is calculated a...,National currency,Billions,Source: Ministry of Finance or Treasury. Minis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2381,-4922,-1521,-4109,-8773,-20584,12934,-4177,5391,-44355,-170879,88127,525762,470137,398089,689750,207885,-521086,-885646,"-1,234.853","2,024.00"
7250,364,VCT,GGSB_NPGDP,St. Vincent and the Grenadines,General government structural balance,The structural budget balance refers to the ge...,Percent of potential GDP,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Filas 2 a 5, solo la segunda columna:


,ISO
1,AFG
2,AFG
3,AFG
4,AFG



len(df): 8626

df.shape: (8626, 61)

df.columns:
['WEO Country Code', 'ISO', 'WEO Subject Code', 'Country', 'Subject Descriptor', 'Subject Notes', 'Units', 'Scale', 'Country/Series-specific Notes', 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 'Estimates Start After']

df.dtypes:
WEO Country Code          object
ISO                       object
WEO Subject Code          object
Country                   object
Subject Descriptor        object
                          ...   
2027                      object
2028                      object
2029                      object
2030                      object
Estimates Start After    float64
Length: 61, dtype: object

df.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8626 en

## 2. API del Banco Mundial (`wbdata`) — PIB real 2010-2023

Países: Estados Unidos, Alemania, Canadá, España, Francia e Italia.
Indicador: `NY.GDP.MKTP.KD` (PIB a precios constantes = PIB real).

In [18]:
# Indicador: PIB real (precios constantes, US$ 2015)
indicador_pib_real = {"NY.GDP.MKTP.KD": "PIB real"}

paises_g6 = ["USA", "DEU", "CAN", "ESP", "FRA", "ITA"]  # códigos ISO3

# El orden es: año (2010), mes (1), día (1)
fecha_inicio = datetime.datetime(2010, 1, 1)
# El orden es: año (2023), mes (1), día (1)
fecha_fin = datetime.datetime(2023, 1, 1)

df_pib_real_g6 = wbdata.get_dataframe(indicador_pib_real, country=paises_g6, date=(fecha_inicio, fecha_fin))
df_pib_real_g6 = df_pib_real_g6.reset_index()  # para que país y fecha queden como columnas normales
df_pib_real_g6.head()

,country,date,PIB real
0,Canada,2023,"1,822,053,687,169.68"
1,Canada,2022,"1,787,149,136,135.45"
2,Canada,2021,"1,706,998,394,665.32"
3,Canada,2020,"1,611,127,783,961.78"
4,Canada,2019,"1,696,606,794,629.52"


In [19]:
explorar_dataset(df_pib_real_g6, "PIB real 2010-2023 (EE.UU., Alemania, Canadá, España, Francia, Italia)")

EXPLORACIÓN: PIB real 2010-2023 (EE.UU., Alemania, Canadá, España, Francia, Italia)

Primeras 5 filas:


,country,date,PIB real
0,Canada,2023,"1,822,053,687,169.68"
1,Canada,2022,"1,787,149,136,135.45"
2,Canada,2021,"1,706,998,394,665.32"
3,Canada,2020,"1,611,127,783,961.78"
4,Canada,2019,"1,696,606,794,629.52"



Últimas 10 filas:


,country,date,PIB real
74,United States,2019,"20,159,639,089,819.00"
75,United States,2018,"19,651,869,117,636.50"
76,United States,2017,"19,085,691,123,385.50"
77,United States,2016,"18,627,887,993,795.50"
78,United States,2015,"18,295,019,000,000.00"
79,United States,2014,"17,771,549,056,393.40"
80,United States,2013,"17,334,068,402,849.70"
81,United States,2012,"16,974,575,729,644.70"
82,United States,2011,"16,594,704,135,237.80"
83,United States,2010,"16,339,094,225,608.80"



3 filas aleatorias:


,country,date,PIB real
42,France,2023,"2,688,358,302,961.22"
75,United States,2018,"19,651,869,117,636.50"
16,Germany,2021,"3,661,544,208,117.20"



Filas 2 a 5, solo la segunda columna:


,date
1,2022
2,2021
3,2020
4,2019



len(df): 84

df.shape: (84, 3)

df.columns:
['country', 'date', 'PIB real']

df.dtypes:
country      object
date         object
PIB real    float64
dtype: object

df.info():
<class 'wbdata.client.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   country   84 non-null     object 
 1   date      84 non-null     object 
 2   PIB real  84 non-null     float64
dtypes: float64(1), object(2)
memory usage: 2.1+ KB



## 3. API del Banco Mundial (`wbdata`) — Brasil 2010-2023

Indicadores solicitados:
- Crecimiento del PIB (% anual): `NY.GDP.MKTP.KD.ZG`
- PIB per cápita en PPA: `NY.GDP.PCAP.PP.KD`
- Tasa de inflación (IPC, % anual): `FP.CPI.TOTL.ZG`
- Tasa de desempleo (% de la fuerza laboral): `SL.UEM.TOTL.ZS`
- Población total: `SP.POP.TOTL`

In [20]:
indicadores_brasil = {
    "NY.GDP.MKTP.KD.ZG": "Crecimiento del PIB",
    "NY.GDP.PCAP.PP.KD": "PIB per cápita PPA",
    "FP.CPI.TOTL.ZG": "Inflación",
    "SL.UEM.TOTL.ZS": "Desempleo",
    "SP.POP.TOTL": "Población",
}

df_brasil = wbdata.get_dataframe(indicadores_brasil, country="BRA", date=(fecha_inicio, fecha_fin))
df_brasil = df_brasil.reset_index()
df_brasil.head()

,date,Crecimiento del PIB,PIB per cápita PPA,Inflación,Desempleo,Población
0,2023,3.24,"19,079.81",4.59,7.95,"211,140,729.00"
1,2022,3.02,"18,554.05",9.28,9.23,"210,306,415.00"
2,2021,4.76,"18,075.71",8.30,13.16,"209,550,294.00"
3,2020,-3.28,"17,327.52",3.21,13.70,"208,660,842.00"
4,2019,1.22,"18,018.62",3.73,11.94,"207,455,459.00"


In [21]:
explorar_dataset(df_brasil, "Brasil 2010-2023: crecimiento PIB, PIB per cápita PPA, inflación, desempleo, población")

EXPLORACIÓN: Brasil 2010-2023: crecimiento PIB, PIB per cápita PPA, inflación, desempleo, población

Primeras 5 filas:


,date,Crecimiento del PIB,PIB per cápita PPA,Inflación,Desempleo,Población
0,2023,3.24,"19,079.81",4.59,7.95,"211,140,729.00"
1,2022,3.02,"18,554.05",9.28,9.23,"210,306,415.00"
2,2021,4.76,"18,075.71",8.30,13.16,"209,550,294.00"
3,2020,-3.28,"17,327.52",3.21,13.70,"208,660,842.00"
4,2019,1.22,"18,018.62",3.73,11.94,"207,455,459.00"



Últimas 10 filas:


,date,Crecimiento del PIB,PIB per cápita PPA,Inflación,Desempleo,Población
4,2019,1.22,"18,018.62",3.73,11.94,"207,455,459.00"
5,2018,1.78,"17,917.75",3.66,12.33,"206,107,261.00"
6,2017,1.32,"17,724.48",3.45,12.79,"204,703,445.00"
7,2016,-3.28,"17,620.93",8.74,11.58,"203,218,114.00"
8,2015,-3.55,"18,357.07",9.03,8.54,"201,675,532.00"
9,2014,0.50,"19,183.17",6.33,6.75,"200,085,127.00"
10,2013,3.00,"19,241.51",6.20,7.07,"198,478,299.00"
11,2012,1.92,"18,832.22",5.40,7.25,"196,876,111.00"
12,2011,3.97,"18,627.81",6.64,7.58,"195,284,734.00"
13,2010,7.53,"18,062.16",5.04,8.42,"193,701,929.00"



3 filas aleatorias:


,date,Crecimiento del PIB,PIB per cápita PPA,Inflación,Desempleo,Población
1,2022,3.02,"18,554.05",9.28,9.23,"210,306,415.00"
7,2016,-3.28,"17,620.93",8.74,11.58,"203,218,114.00"
4,2019,1.22,"18,018.62",3.73,11.94,"207,455,459.00"



Filas 2 a 5, solo la segunda columna:


,Crecimiento del PIB
1,3.02
2,4.76
3,-3.28
4,1.22



len(df): 14

df.shape: (14, 6)

df.columns:
['date', 'Crecimiento del PIB', 'PIB per cápita PPA', 'Inflación', 'Desempleo', 'Población']

df.dtypes:
date                    object
Crecimiento del PIB    float64
PIB per cápita PPA     float64
Inflación              float64
Desempleo              float64
Población              float64
dtype: object

df.info():
<class 'wbdata.client.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 14 non-null     object 
 1   Crecimiento del PIB  14 non-null     float64
 2   PIB per cápita PPA   14 non-null     float64
 3   Inflación            14 non-null     float64
 4   Desempleo            14 non-null     float64
 5   Población            14 non-null     float64
dtypes: float64(5), object(1)
memory usage: 804.0+ bytes



## 4. Datos abiertos de Colombia (datos.gov.co) vía API

Mismo procedimiento visto en clase: entrar al dataset en https://www.datos.gov.co , ir a **Exportar → API de acceso** y copiar el endpoint.

Se traen dos conjuntos de datos:
- Educación (`MEN_ESTADISTICAS_EN_EDUCACION...`, visto en clase)
- Casos positivos de COVID-19 en Colombia (visto en clase)

*(Se puede reemplazar cualquiera de los dos IDs por otro dataset que llame más la atención; el procedimiento es idéntico.)*

In [22]:
# Dataset 1: Estadísticas de educación en preescolar, básica y media por departamento (MEN)
url_1 = "https://www.datos.gov.co/resource/ji8i-4anb.json"

response_1 = requests.get(url_1)
data_1 = response_1.json()

df_educacion = pd.DataFrame(data_1)
df_educacion.head()

,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,cobertura_neta_transicion,cobertura_neta_primaria,cobertura_neta_secundaria,cobertura_neta_media,cobertura_bruta,cobertura_bruta_transicion,cobertura_bruta_primaria,cobertura_bruta_secundaria,cobertura_bruta_media,tamano_promedio_grupo,sedes_conectadas_a_internet,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,aprobacion,aprobacion_transicion,aprobacion_primaria,aprobacion_secundaria,aprobacion_media,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
0,2011,5,Antioquia,1288473,94.01,93.85,70.28,94.12,75.68,44.37,106.81,85.69,117.5,110.81,84.06,27.47,74.48,3.97,3.62,3.65,4.57,3.71,93.98,0.07,94.56,4.57,93.33,2.06,0.07,94.56,2.54,2.96,4.25,0.07,4.56,5.27,1.68
1,2011,8,Atlántico,523935,99.32,99.05,50.59,98.93,80.22,50.17,107.88,84.19,120.41,107.89,87.98,24.42,80.46,2.76,2.6,3.06,2.42,2.61,96.7,0.12,96.49,2.42,96.64,0.54,0.12,96.49,0.67,0.75,1.82,0.12,1.77,2.18,0.88
2,2011,11,"Bogotá, D.C.",1479334,90.7,90.29,68.63,86.99,84.7,55.01,97.78,82.04,97.94,106.48,87.76,26.75,94.21,3.95,10.05,5.3,1.96,2.55,96.05,0,94.69,1.96,97.45,0,0,94.69,0,0,3.23,0,2.3,5.11,2.57
3,2011,13,Bolívar,496676,91.57,91.4,59.74,90.81,67.34,39.17,110.41,98.82,126.45,108.15,80.64,20.8,30.12,3.14,1.85,2.93,3.79,3.13,94.76,0.46,95.48,3.79,93.2,2.1,0.46,95.48,2.75,3.67,4.43,0.46,4.44,5.37,2.28
4,2011,15,Boyacá,300501,86.16,86.11,63.36,82.5,74.65,49.09,104.15,78.87,99.88,119.78,94.76,22.77,25.26,3.07,2.4,2.24,4.03,3.51,94.2,0.17,96.1,4.03,93.23,2.73,0.17,96.1,4.31,3.26,2.62,0.17,1.9,4.19,1.55


In [23]:
explorar_dataset(df_educacion, "Datos abiertos Colombia — Estadísticas de educación (MEN)")

EXPLORACIÓN: Datos abiertos Colombia — Estadísticas de educación (MEN)

Primeras 5 filas:


,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,cobertura_neta_transicion,cobertura_neta_primaria,cobertura_neta_secundaria,cobertura_neta_media,cobertura_bruta,cobertura_bruta_transicion,cobertura_bruta_primaria,cobertura_bruta_secundaria,cobertura_bruta_media,tamano_promedio_grupo,sedes_conectadas_a_internet,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,aprobacion,aprobacion_transicion,aprobacion_primaria,aprobacion_secundaria,aprobacion_media,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
0,2011,5,Antioquia,1288473,94.01,93.85,70.28,94.12,75.68,44.37,106.81,85.69,117.5,110.81,84.06,27.47,74.48,3.97,3.62,3.65,4.57,3.71,93.98,0.07,94.56,4.57,93.33,2.06,0.07,94.56,2.54,2.96,4.25,0.07,4.56,5.27,1.68
1,2011,8,Atlántico,523935,99.32,99.05,50.59,98.93,80.22,50.17,107.88,84.19,120.41,107.89,87.98,24.42,80.46,2.76,2.6,3.06,2.42,2.61,96.7,0.12,96.49,2.42,96.64,0.54,0.12,96.49,0.67,0.75,1.82,0.12,1.77,2.18,0.88
2,2011,11,"Bogotá, D.C.",1479334,90.7,90.29,68.63,86.99,84.7,55.01,97.78,82.04,97.94,106.48,87.76,26.75,94.21,3.95,10.05,5.3,1.96,2.55,96.05,0,94.69,1.96,97.45,0,0,94.69,0,0,3.23,0,2.3,5.11,2.57
3,2011,13,Bolívar,496676,91.57,91.4,59.74,90.81,67.34,39.17,110.41,98.82,126.45,108.15,80.64,20.8,30.12,3.14,1.85,2.93,3.79,3.13,94.76,0.46,95.48,3.79,93.2,2.1,0.46,95.48,2.75,3.67,4.43,0.46,4.44,5.37,2.28
4,2011,15,Boyacá,300501,86.16,86.11,63.36,82.5,74.65,49.09,104.15,78.87,99.88,119.78,94.76,22.77,25.26,3.07,2.4,2.24,4.03,3.51,94.2,0.17,96.1,4.03,93.23,2.73,0.17,96.1,4.31,3.26,2.62,0.17,1.9,4.19,1.55



Últimas 10 filas:


,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,cobertura_neta_transicion,cobertura_neta_primaria,cobertura_neta_secundaria,cobertura_neta_media,cobertura_bruta,cobertura_bruta_transicion,cobertura_bruta_primaria,cobertura_bruta_secundaria,cobertura_bruta_media,tamano_promedio_grupo,sedes_conectadas_a_internet,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,aprobacion,aprobacion_transicion,aprobacion_primaria,aprobacion_secundaria,aprobacion_media,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
452,2024,76,Valle del Cauca,836702,76.03,75.84,54.51,73.57,65.01,45.72,84.85,72.5,86.13,87.09,83.14,NaN,NaN,4.84,5.34,4.27,5.92,3.61,87.62,0.39,90.12,5.92,90.53,7.54,4.89,90.12,12.01,5.86,8.84,4.89,8.34,11.98,4.38
453,2024,81,Arauca,67864,88.81,88.76,64.23,87.73,68.21,40.82,99.66,96.3,109.04,95.78,84.58,NaN,NaN,5.27,5.51,5.16,5.86,3.83,86.86,0.24,87.69,5.86,91,7.87,3.49,87.69,12.01,5.18,9.14,3.49,9.67,11.49,4.03
454,2024,85,Casanare,96647,91.04,91.02,68.93,89,80.24,50.86,99.01,86.08,101.86,105.92,84.22,NaN,NaN,3.77,4,3.34,4.66,2.67,89.46,0.1,92.52,4.66,91.6,6.77,1.64,92.52,11.93,5.73,7.78,1.64,7.05,11.57,3.26
455,2024,86,Putumayo,81100,80.03,80.01,50.77,78.12,66.49,38.12,90.06,83.06,94.52,94.34,73.94,NaN,NaN,6.49,5.6,4.89,9.24,5.36,87.44,0.14,90.68,9.24,89.82,6.07,3.02,90.68,10.04,4.82,8.5,3.02,7.56,12.33,4.57
456,2024,88,"Archipiélago de San Andrés, Providencia y Sant...",10726,94.06,94,78.05,93.57,82.79,54.3,99.57,93.17,104.39,102.98,82.93,NaN,NaN,1.94,0.71,1.71,2.26,2.51,88.25,0,92.1,2.26,89.23,9.81,0.36,92.1,16.99,8.26,8.45,0.36,7.5,12.87,4.36
457,2024,91,Amazonas,23217,74.82,74.82,49.71,76.87,54.54,26.37,86.84,81.03,103.75,81.23,58.8,NaN,NaN,5.11,6.95,2.95,7.74,6.2,90.44,0.76,91.82,7.74,90.76,4.45,5.51,91.82,4.58,3.04,12.83,5.51,13.19,15.77,7.39
458,2024,94,Guainía,16738,71.94,71.94,58.79,72.04,37.34,11.28,82.34,88.55,107.35,69.65,41.62,NaN,NaN,4.73,4.84,3.87,6.69,3.71,80.36,1.47,80.23,6.69,85.05,14.91,5.02,80.23,18.19,11.24,14.82,5.02,17.19,15.55,5.62
459,2024,95,Guaviare,23189,74.16,74.11,48.68,69.12,59,31.95,87.91,81.47,93.86,90.25,71.08,NaN,NaN,4.8,6.83,3.93,6.23,2.75,87.14,3.98,89.53,6.23,92.11,8.06,6.33,89.53,12.32,5.14,11.13,6.33,11.08,14.9,3.05
460,2024,97,Vaupés,15344,53.45,53.45,35.5,51.87,36.99,14.24,63.45,55.23,73.81,62.71,44.3,NaN,NaN,5.34,3.75,3.74,7.5,7,87.34,0,90.46,7.5,84.9,7.32,6.88,90.46,10.86,8.1,16,6.88,19.22,15.17,9.4
461,2024,99,Vichada,33641,68.28,68.25,52.78,76.39,30.55,10.46,75.93,75.24,111.78,55.31,28.65,NaN,NaN,6.31,4.05,6.26,7.53,4.83,80.87,0,77.22,7.53,91.16,12.81,0.91,77.22,9.77,4.01,17.06,0.91,22.17,12.59,4.08



3 filas aleatorias:


,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,cobertura_neta_transicion,cobertura_neta_primaria,cobertura_neta_secundaria,cobertura_neta_media,cobertura_bruta,cobertura_bruta_transicion,cobertura_bruta_primaria,cobertura_bruta_secundaria,cobertura_bruta_media,tamano_promedio_grupo,sedes_conectadas_a_internet,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,aprobacion,aprobacion_transicion,aprobacion_primaria,aprobacion_secundaria,aprobacion_media,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
139,2015,19,Cauca,312453,82.21,82.21,44.7,84.42,63.04,31.28,99.04,80.51,112.85,99.32,74.71,19.12,44.93,2.17,2.12,1.31,3.43,2.46,91.64,1.03,92.86,3.43,92.05,6.19,1.03,92.86,8.17,5.48,1.23,1.03,1.24,1.6,0.68
221,2017,76,Valle del Cauca,873860,80.4,80.21,49.95,76.38,71.03,40.74,91.94,75.77,93.77,100.98,77.74,28.88,47.05,3.71,4.24,3.11,4.48,3.24,89.59,0.84,91.59,4.48,91.98,6.7,0.84,91.59,10.03,4.78,1.45,0.84,1.35,1.93,0.78
332,2021,11,"Bogotá, D,C,",1174274,94.28,93.80,63.99,89.92,89.19,58.84,102.57,79.21,101.47,111.35,99.55,NaN,NaN,1.29,1.07,1.08,1.35,1.88,88.15,0.86,91.46,1.35,87.15,10.56,0.86,91.46,15.84,10.97,3.29,0.86,2.44,5.20,2.39



Filas 2 a 5, solo la segunda columna:


,c_digo_departamento
1,8
2,11
3,13
4,15



len(df): 462

df.shape: (462, 37)

df.columns:
['ano', 'c_digo_departamento', 'departamento', 'poblacion_5_16', 'tasa_matriculacion_5_16', 'cobertura_neta', 'cobertura_neta_transicion', 'cobertura_neta_primaria', 'cobertura_neta_secundaria', 'cobertura_neta_media', 'cobertura_bruta', 'cobertura_bruta_transicion', 'cobertura_bruta_primaria', 'cobertura_bruta_secundaria', 'cobertura_bruta_media', 'tamano_promedio_grupo', 'sedes_conectadas_a_internet', 'desercion', 'desercion_transicion', 'desercion_primaria', 'desercion_secundaria', 'desercion_media', 'aprobacion', 'aprobacion_transicion', 'aprobacion_primaria', 'aprobacion_secundaria', 'aprobacion_media', 'reprobacion', 'reprobacion_transicion', 'reprobacion_primaria', 'reprobacion_secundaria', 'reprobacion_media', 'repitencia', 'repitencia_transicion', 'repitencia_primaria', 'repitencia_secundaria', 'repitencia_media']

df.dtypes:
ano                            object
c_digo_departamento            object
departamento                 

In [24]:
# Dataset 2: Casos positivos de COVID-19 en Colombia
url_2 = "https://www.datos.gov.co/resource/gt2j-8ykr.json"

response_2 = requests.get(url_2)
data_2 = response_2.json()

df_covid = pd.DataFrame(data_2)
df_covid.head()

,fecha_reporte_web,id_de_caso,fecha_de_notificaci_n,departamento,departamento_nom,ciudad_municipio,ciudad_municipio_nom,edad,unidad_medida,sexo,fuente_tipo_contagio,ubicacion,estado,recuperado,fecha_inicio_sintomas,fecha_diagnostico,fecha_recuperado,tipo_recuperacion,per_etn_,fecha_muerte,nom_grupo_
0,2020-12-24 00:00:00,1556979,2020-12-22 00:00:00,76,VALLE,76001,CALI,67,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-21 00:00:00,2020-12-23 00:00:00,2021-01-04 00:00:00,Tiempo,6,NaN,NaN
1,2020-12-24 00:00:00,1556980,2020-12-19 00:00:00,76,VALLE,76001,CALI,66,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-07 00:00:00,2020-12-23 00:00:00,2020-12-25 00:00:00,Tiempo,6,NaN,NaN
2,2020-12-24 00:00:00,1556981,2020-12-19 00:00:00,76,VALLE,76001,CALI,68,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-18 00:00:00,2020-12-22 00:00:00,2021-01-01 00:00:00,Tiempo,6,NaN,NaN
3,2020-12-24 00:00:00,1556982,2020-12-22 00:00:00,76,VALLE,76001,CALI,74,1,F,Comunitaria,Fallecido,Fallecido,Fallecido,2020-12-17 00:00:00,2020-12-23 00:00:00,NaN,NaN,6,2020-12-30 00:00:00,NaN
4,2020-12-24 00:00:00,1556983,2020-12-22 00:00:00,76,VALLE,76001,CALI,65,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-21 00:00:00,2020-12-23 00:00:00,2021-01-04 00:00:00,Tiempo,6,NaN,NaN


In [25]:
explorar_dataset(df_covid, "Datos abiertos Colombia — Casos positivos de COVID-19")

EXPLORACIÓN: Datos abiertos Colombia — Casos positivos de COVID-19

Primeras 5 filas:


,fecha_reporte_web,id_de_caso,fecha_de_notificaci_n,departamento,departamento_nom,ciudad_municipio,ciudad_municipio_nom,edad,unidad_medida,sexo,fuente_tipo_contagio,ubicacion,estado,recuperado,fecha_inicio_sintomas,fecha_diagnostico,fecha_recuperado,tipo_recuperacion,per_etn_,fecha_muerte,nom_grupo_
0,2020-12-24 00:00:00,1556979,2020-12-22 00:00:00,76,VALLE,76001,CALI,67,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-21 00:00:00,2020-12-23 00:00:00,2021-01-04 00:00:00,Tiempo,6,NaN,NaN
1,2020-12-24 00:00:00,1556980,2020-12-19 00:00:00,76,VALLE,76001,CALI,66,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-07 00:00:00,2020-12-23 00:00:00,2020-12-25 00:00:00,Tiempo,6,NaN,NaN
2,2020-12-24 00:00:00,1556981,2020-12-19 00:00:00,76,VALLE,76001,CALI,68,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-18 00:00:00,2020-12-22 00:00:00,2021-01-01 00:00:00,Tiempo,6,NaN,NaN
3,2020-12-24 00:00:00,1556982,2020-12-22 00:00:00,76,VALLE,76001,CALI,74,1,F,Comunitaria,Fallecido,Fallecido,Fallecido,2020-12-17 00:00:00,2020-12-23 00:00:00,NaN,NaN,6,2020-12-30 00:00:00,NaN
4,2020-12-24 00:00:00,1556983,2020-12-22 00:00:00,76,VALLE,76001,CALI,65,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-21 00:00:00,2020-12-23 00:00:00,2021-01-04 00:00:00,Tiempo,6,NaN,NaN



Últimas 10 filas:


,fecha_reporte_web,id_de_caso,fecha_de_notificaci_n,departamento,departamento_nom,ciudad_municipio,ciudad_municipio_nom,edad,unidad_medida,sexo,fuente_tipo_contagio,ubicacion,estado,recuperado,fecha_inicio_sintomas,fecha_diagnostico,fecha_recuperado,tipo_recuperacion,per_etn_,fecha_muerte,nom_grupo_
990,2020-09-03 00:00:00,640268,2020-08-13 00:00:00,68,SANTANDER,68307,GIRON,23,1,M,Comunitaria,Casa,Leve,Recuperado,2020-08-08 00:00:00,2020-08-24 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
991,2020-09-03 00:00:00,640269,2020-08-14 00:00:00,68,SANTANDER,68081,BARRANCABERMEJA,45,1,F,Comunitaria,Casa,Leve,Recuperado,2020-08-09 00:00:00,2020-08-25 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
992,2020-09-03 00:00:00,640270,2020-08-11 00:00:00,5,ANTIOQUIA,5088,BELLO,34,1,M,Relacionado,Casa,Leve,Recuperado,2020-08-07 00:00:00,2020-08-22 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
993,2020-09-03 00:00:00,640271,2020-08-12 00:00:00,68,SANTANDER,68276,FLORIDABLANCA,56,1,M,Comunitaria,Casa,Leve,Recuperado,2020-08-07 00:00:00,2020-08-23 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
994,2020-09-03 00:00:00,640272,2020-08-14 00:00:00,68,SANTANDER,68081,BARRANCABERMEJA,27,1,F,Comunitaria,Casa,Leve,Recuperado,2020-08-09 00:00:00,2020-08-25 00:00:00,2020-09-04 00:00:00,PCR,6,NaN,NaN
995,2020-09-03 00:00:00,640273,2020-08-11 00:00:00,5,ANTIOQUIA,5088,BELLO,37,1,F,Comunitaria,Casa,Leve,Recuperado,2020-08-07 00:00:00,2020-08-22 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
996,2020-09-03 00:00:00,640274,2020-08-13 00:00:00,68,SANTANDER,68081,BARRANCABERMEJA,12,1,F,Comunitaria,Casa,Leve,Recuperado,2020-08-08 00:00:00,2020-08-24 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
997,2020-09-03 00:00:00,640275,2020-08-13 00:00:00,68,SANTANDER,68081,BARRANCABERMEJA,11,1,M,Comunitaria,Casa,Leve,Recuperado,2020-08-08 00:00:00,2020-08-24 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
998,2020-09-03 00:00:00,640276,2020-08-13 00:00:00,68,SANTANDER,68081,BARRANCABERMEJA,6,1,F,Comunitaria,Casa,Leve,Recuperado,2020-08-08 00:00:00,2020-08-24 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN
999,2020-09-03 00:00:00,640277,2020-08-14 00:00:00,5,ANTIOQUIA,5088,BELLO,31,1,M,Comunitaria,Casa,Leve,Recuperado,2020-08-10 00:00:00,2020-08-25 00:00:00,2020-09-04 00:00:00,Tiempo,6,NaN,NaN



3 filas aleatorias:


,fecha_reporte_web,id_de_caso,fecha_de_notificaci_n,departamento,departamento_nom,ciudad_municipio,ciudad_municipio_nom,edad,unidad_medida,sexo,fuente_tipo_contagio,ubicacion,estado,recuperado,fecha_inicio_sintomas,fecha_diagnostico,fecha_recuperado,tipo_recuperacion,per_etn_,fecha_muerte,nom_grupo_
227,2021-01-12 00:00:00,1814988,2020-12-20 00:00:00,73,TOLIMA,73001,IBAGUE,51,1,F,Comunitaria,Casa,Leve,Recuperado,2020-12-17 00:00:00,2020-12-31 00:00:00,2021-01-13 00:00:00,Tiempo,6,NaN,NaN
471,2020-09-23 00:00:00,781861,2020-09-11 00:00:00,8,ATLANTICO,8758,SOLEDAD,80,1,F,Comunitaria,Casa,Leve,Recuperado,2020-09-09 00:00:00,2020-09-22 00:00:00,2020-09-28 00:00:00,Tiempo,6,NaN,NaN
245,2020-05-23 00:00:00,20156,2020-05-15 00:00:00,13001,CARTAGENA,13001,CARTAGENA,43,1,M,Relacionado,Casa,Leve,Recuperado,2020-05-15 00:00:00,2020-05-23 00:00:00,2020-06-18 00:00:00,Tiempo,6,NaN,NaN



Filas 2 a 5, solo la segunda columna:


,id_de_caso
1,1556980
2,1556981
3,1556982
4,1556983



len(df): 1000

df.shape: (1000, 21)

df.columns:
['fecha_reporte_web', 'id_de_caso', 'fecha_de_notificaci_n', 'departamento', 'departamento_nom', 'ciudad_municipio', 'ciudad_municipio_nom', 'edad', 'unidad_medida', 'sexo', 'fuente_tipo_contagio', 'ubicacion', 'estado', 'recuperado', 'fecha_inicio_sintomas', 'fecha_diagnostico', 'fecha_recuperado', 'tipo_recuperacion', 'per_etn_', 'fecha_muerte', 'nom_grupo_']

df.dtypes:
fecha_reporte_web        object
id_de_caso               object
fecha_de_notificaci_n    object
departamento             object
departamento_nom         object
ciudad_municipio         object
ciudad_municipio_nom     object
edad                     object
unidad_medida            object
sexo                     object
fuente_tipo_contagio     object
ubicacion                object
estado                   object
recuperado               object
fecha_inicio_sintomas    object
fecha_diagnostico        object
fecha_recuperado         object
tipo_recuperacion        objec

## 5. Otra fuente de datos económicos/sociales vía API: FRED (Banco de la Reserva Federal de San Luis)

**FRED** (Federal Reserve Economic Data) publica series económicas y sociales de EE. UU. y otros países mediante una API REST gratuita.

**Pasos:**
1. Crear una cuenta gratuita en https://fred.stlouisfed.org/
2. Solicitar una API key gratuita en https://fred.stlouisfed.org/docs/api/api_key.html
3. Reemplazar `TU_API_KEY` por la llave obtenida.

Se usa la misma lógica de `requests` + `.json()` + `pd.DataFrame()` que en el punto anterior. Serie de ejemplo: `GDPC1` (PIB real trimestral de EE. UU.).

In [32]:
API_KEY_FRED = "TU_API_KEY"  # <-- reemplazar por la API key gratuita de FRED
serie_fred = "GDPC1"  # PIB real de EE. UU. (trimestral)

url_fred = "https://api.stlouisfed.org/fred/series/observations"
params_fred = {
    "series_id": serie_fred,
    "api_key": "a771616ca5b1f674c3f00a5ebf70d69f",
    "file_type": "json",
}

response_fred = requests.get(url_fred, params=params_fred)

# Verificar si la solicitud fue exitosa
if response_fred.status_code == 200:
    data_fred = response_fred.json()["observations"]
    df_fred = pd.DataFrame(data_fred)
    df_fred = df_fred[["date", "value"]].rename(columns={"date": "fecha", "value": "pib_real_eeuu"})
    df_fred.head()
else:
    print(f"Error al obtener datos de FRED: {response_fred.status_code}")
    print(response_fred.json()) # Imprimir la respuesta JSON completa para depuración

In [33]:
explorar_dataset(df_fred, "FRED — PIB real trimestral de EE.UU. (GDPC1)")

EXPLORACIÓN: FRED — PIB real trimestral de EE.UU. (GDPC1)

Primeras 5 filas:


,fecha,pib_real_eeuu
0,1947-01-01,2182.681
1,1947-04-01,2176.892
2,1947-07-01,2172.432
3,1947-10-01,2206.452
4,1948-01-01,2239.682



Últimas 10 filas:


,fecha,pib_real_eeuu
308,2024-01-01,23082.119
309,2024-04-01,23286.508
310,2024-07-01,23478.57
311,2024-10-01,23586.542
312,2025-01-01,23548.21
313,2025-04-01,23770.976
314,2025-07-01,24026.834
315,2025-10-01,24055.749
316,2026-01-01,24180.419
317,2026-04-01,24269.613



3 filas aleatorias:


,fecha,pib_real_eeuu
191,1994-10-01,11279.932
15,1950-10-01,2559.214
230,2004-07-01,15512.619



Filas 2 a 5, solo la segunda columna:


,pib_real_eeuu
1,2176.892
2,2172.432
3,2206.452
4,2239.682



len(df): 318

df.shape: (318, 2)

df.columns:
['fecha', 'pib_real_eeuu']

df.dtypes:
fecha            object
pib_real_eeuu    object
dtype: object

df.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   fecha          318 non-null    object
 1   pib_real_eeuu  318 non-null    object
dtypes: object(2)
memory usage: 5.1+ KB



## Conclusión

Se importaron y exploraron 8 conjuntos de datos desde 4 fuentes distintas: la base WEO del FMI (desde Google Drive), la API del Banco Mundial vía `wbdata` (PIB real de 6 países y 5 indicadores para Brasil), dos conjuntos de datos abiertos de Colombia (Socrata) y la API de FRED, todos con la misma rutina de inspección inicial vista en clase.

In [34]:
def verificar_punto(nombre_punto, condiciones):
    """
    Recibe el nombre del punto y una lista de tuplas (descripcion, booleano).
    Imprime el resultado de cada condición y devuelve True si todas pasaron.
    """
    print(f"\n{'='*80}\nPUNTO: {nombre_punto}\n{'='*80}")
    todo_ok = True
    for descripcion, resultado in condiciones:
        marca = "✅" if resultado else "❌"
        print(f"  {marca}  {descripcion}")
        if not resultado:
            todo_ok = False
    print(f"\n  --> {nombre_punto}: {'TODO CORRECTO ✅' if todo_ok else 'REVISAR ❌'}")
    return todo_ok


resultados_generales = {}

In [35]:
# --- Punto 1: WEO ---
try:
    condiciones = [
        ("df_weo existe y no está vacío", "df_weo" in globals() and len(df_weo) > 0),
        ("df_weo tiene más de 5 columnas", "df_weo" in globals() and df_weo.shape[1] > 5),
    ]
except NameError:
    condiciones = [("df_weo existe", False)]

resultados_generales["1. WEO (FMI)"] = verificar_punto("1. Base WEO del FMI", condiciones)


PUNTO: 1. Base WEO del FMI
  ✅  df_weo existe y no está vacío
  ✅  df_weo tiene más de 5 columnas

  --> 1. Base WEO del FMI: TODO CORRECTO ✅


In [36]:
# --- Punto 2: PIB real 6 países (Banco Mundial) ---
try:
    paises_esperados = {"United States", "Germany", "Canada", "Spain", "France", "Italy"}
    paises_obtenidos = set(df_pib_real_g6["country"].unique()) if "country" in df_pib_real_g6.columns else set()

    condiciones = [
        ("df_pib_real_g6 existe y no está vacío", "df_pib_real_g6" in globals() and len(df_pib_real_g6) > 0),
        ("Contiene los 6 países esperados", len(paises_esperados & paises_obtenidos) == 6 or len(df_pib_real_g6) >= 6*10),
        ("No hay valores nulos en la columna de PIB real",
         df_pib_real_g6["PIB real"].isna().sum() == 0 if "PIB real" in df_pib_real_g6.columns else False),
    ]
except NameError:
    condiciones = [("df_pib_real_g6 existe", False)]

resultados_generales["2. PIB real G6 (Banco Mundial)"] = verificar_punto("2. PIB real 2010-2023 (6 países)", condiciones)


PUNTO: 2. PIB real 2010-2023 (6 países)
  ✅  df_pib_real_g6 existe y no está vacío
  ✅  Contiene los 6 países esperados
  ✅  No hay valores nulos en la columna de PIB real

  --> 2. PIB real 2010-2023 (6 países): TODO CORRECTO ✅


In [37]:
# --- Punto 3: Brasil (Banco Mundial) ---
try:
    columnas_esperadas_brasil = {"Crecimiento del PIB", "PIB per cápita PPA", "Inflación", "Desempleo", "Población"}
    columnas_obtenidas_brasil = set(df_brasil.columns)

    condiciones = [
        ("df_brasil existe y no está vacío", "df_brasil" in globals() and len(df_brasil) > 0),
        ("Contiene los 5 indicadores solicitados",
         columnas_esperadas_brasil.issubset(columnas_obtenidas_brasil)),
        ("Cubre 14 años (2010-2023)", len(df_brasil) >= 14),
    ]
except NameError:
    condiciones = [("df_brasil existe", False)]

resultados_generales["3. Indicadores Brasil (Banco Mundial)"] = verificar_punto("3. Brasil 2010-2023", condiciones)


PUNTO: 3. Brasil 2010-2023
  ✅  df_brasil existe y no está vacío
  ✅  Contiene los 5 indicadores solicitados
  ✅  Cubre 14 años (2010-2023)

  --> 3. Brasil 2010-2023: TODO CORRECTO ✅


In [38]:
# --- Punto 4: Datos abiertos de Colombia ---
try:
    condiciones = [
        ("df_educacion existe y no está vacío", "df_educacion" in globals() and len(df_educacion) > 0),
        ("df_covid existe y no está vacío", "df_covid" in globals() and len(df_covid) > 0),
        ("Los dos datasets son distintos entre sí",
         "df_educacion" in globals() and "df_covid" in globals()
         and list(df_educacion.columns) != list(df_covid.columns)),
    ]
except NameError:
    condiciones = [("Datasets de Colombia existen", False)]

resultados_generales["4. Datos abiertos Colombia (2 datasets)"] = verificar_punto("4. Datos abiertos de Colombia", condiciones)


PUNTO: 4. Datos abiertos de Colombia
  ✅  df_educacion existe y no está vacío
  ✅  df_covid existe y no está vacío
  ✅  Los dos datasets son distintos entre sí

  --> 4. Datos abiertos de Colombia: TODO CORRECTO ✅


In [40]:
# --- Punto 5: Otra API (FRED) ---
try:
    condiciones = [
        ("df_fred existe y no está vacío", "df_fred" in globals() and len(df_fred) > 0),
        ("Se usó una API key real (no quedó 'TU_API_KEY')",
         "params_fred" in globals() and params_fred.get("api_key") not in (None, "TU_API_KEY")),
        ("df_fred tiene columna de fecha y de valor", set(["fecha", "pib_real_eeuu"]).issubset(df_fred.columns)),
    ]
except NameError:
    condiciones = [("df_fred existe", False)]

resultados_generales["5. Otra API (FRED)"] = verificar_punto("5. Otra fuente de datos vía API", condiciones)


PUNTO: 5. Otra fuente de datos vía API
  ✅  df_fred existe y no está vacío
  ✅  Se usó una API key real (no quedó 'TU_API_KEY')
  ✅  df_fred tiene columna de fecha y de valor

  --> 5. Otra fuente de datos vía API: TODO CORRECTO ✅
